<a href="https://colab.research.google.com/github/rijosudu123/ict-assessment/blob/main/sample_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import time

In [29]:
DATA_PATH = "/content/Crimes_-_2015_20260702.csv.xls"
OUT_DIR = "Outputs"
INTERVAL_SECONDS = 20
MAX_RUNS =3
os.makedirs(OUT_DIR, exist_ok=True)

In [30]:
def extract () :
  crimes_data = pd.read_csv(DATA_PATH)
  crimes_data_len = len(crimes_data)
  raw_file_path = os.path.join(OUT_DIR, "raw_crimes_data.csv")
  crimes_data.to_csv(raw_file_path, index=False)
  print(f"Raw data saved to: {raw_file_path}")
  print(f"Extracted {crimes_data_len} rows")
  return crimes_data

In [31]:
def transform(crimes_data) :
  crimes_samples = crimes_data.sample(n=10000, random_state = 46).reset_index(drop = True)
  crimes_samples["Extracted at"] = datetime.now()
  crimes_samples.drop_duplicates(inplace = True)
  crimes_samples['Date'] = pd.to_datetime(crimes_samples['Date'])
  crimes_samples['Crime_Date'] = crimes_samples['Date'].dt.date
  crimes_samples['Crime_Time'] = crimes_samples['Date'].dt.time
  crimes_samples[['Block_Number', 'Direction', 'Street']] = crimes_samples['Block'].str.extract(r'(\S+)\s+([NSEW])\s+(.+)')
  crimes_samples.drop(columns=['Date','Block','Updated On','X Coordinates','Y Coordinates','Year','Location'],errors = 'ignore', inplace=True)
  crimes_samples['Location Description'] = crimes_samples['Location Description'].fillna('UNKNOWN')
  crimes_samples.dropna(subset = ['Ward','Community Area','Latitude','Longitude'],inplace = True)
  print(f"Transformed {len(crimes_samples)} rows")
  return crimes_samples


In [32]:
def load(crimes_data) :
  timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
  cleaned_path = os.path.join(OUT_DIR,f"cleaned_crimes_{timestamp}.csv")
  crimes_data.to_csv(cleaned_path, index=False)
  summary = pd.DataFrame({
      "metric" : ['Total Records','Total Arrests','Domestic Crimes'],
      "value" : [len(crimes_data),crimes_data["Arrest"].sum(),crimes_data["Domestic"].sum()]
  })

  filename = datetime.now().strftime("%Y%m%d_%H%M%S")
  load_path = f"{OUT_DIR}/summary_{filename}.csv"

  summary.to_csv(load_path,index = False)
  print(f"Loaded {len(crimes_data)} rows")

In [33]:
def run_pipeline():
  crimes_data = extract()
  cleaned_data = transform(crimes_data)
  load(cleaned_data)
  print("pipeline completed")

In [34]:
def schedule_pipeline():
  run_count = 0
  while run_count<MAX_RUNS:
    run_pipeline()
    run_count += 1

    if run_count < MAX_RUNS :
      print(f"next run will happen after {INTERVAL_SECONDS}")
      time.sleep(INTERVAL_SECONDS)

  print("All scheduled runs are completed")


if __name__ == '__main__':
  schedule_pipeline()

Raw data saved to: Outputs/raw_crimes_data.csv
Extracted 264902 rows


/tmp/ipykernel_5593/747927354.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crimes_samples['Date'] = pd.to_datetime(crimes_samples['Date'])


Transformed 9749 rows
Loaded 9749 rows
pipeline completed
next run will happen after 20
Raw data saved to: Outputs/raw_crimes_data.csv
Extracted 264902 rows


/tmp/ipykernel_5593/747927354.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crimes_samples['Date'] = pd.to_datetime(crimes_samples['Date'])


Transformed 9749 rows
Loaded 9749 rows
pipeline completed
next run will happen after 20
Raw data saved to: Outputs/raw_crimes_data.csv
Extracted 264902 rows


/tmp/ipykernel_5593/747927354.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crimes_samples['Date'] = pd.to_datetime(crimes_samples['Date'])


Transformed 9749 rows
Loaded 9749 rows
pipeline completed
All scheduled runs are completed
